# RetailHub Data Processing & Analysis

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
HIGH_VALUE_THRESHOLD = 3000
NORTH_INDIAN_CITIES = ["Delhi", "Jaipur", "Lucknow", "Bhopal", "Agra"]
SOUTH_INDIAN_CITIES = ["Bangalore", "Hyderabad", "Chennai", "Kochi", "Visakhapatnam"]

## Phase 1 — Ingestion & Schema Definition

- Read all three CSV files into DataFrames, explicitly defining schemas rather than inferring them
- Handle the malformed records in orders.csv gracefully by capturing bad rows into a separate quarantine DataFrame instead of crashing the pipeline

In [0]:
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_ids", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("shipping_date", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("status", StringType(), True),
    StructField("discount_tags", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])

raw_orders_df = spark.read.format("csv")\
                .option("header", True)\
                .option("mode", "PERMISSIVE")\
                .option("columnNameOfCorruptRecord", "_corrupt_record")\
                .schema(orders_schema)\
                .load("/Volumes/workspace/learning_spark/spark/mini_projects_dataset/project_3/orders.csv")

display(raw_orders_df)

In [0]:
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("FullName", StringType(), True),
    StructField("EMAIL", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("Phone", StringType(), True),
    StructField("registration_date", StringType(), True),
    StructField("_corrupt_record", StringType(), True),
])

raw_customers_df = spark.read.format("csv")\
                    .option("header", True)\
                    .option("mode", "PERMISSIVE")\
                    .option("columnNameOfCorruptRecord", "_corrupt_record")\
                    .schema(customers_schema)\
                    .load("/Volumes/workspace/learning_spark/spark/mini_projects_dataset/project_3/customers.csv")

display(raw_customers_df)

In [0]:
products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("ProductName", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Price", StringType(), True),
    StructField("tags", StringType(), True),
    StructField("_corrupt_record", StringType(), True)
])

raw_products_df = spark.read.format("csv")\
                    .option("header", True)\
                    .option("mode", "PERMISSIVE")\
                    .option("columnNameOfCorruptRecord", "_corrupt_record")\
                    .schema(products_schema)\
                    .load("/Volumes/workspace/learning_spark/spark/mini_projects_dataset/project_3/products.csv")

display(raw_products_df)

In [0]:
quarantine_orders_df = raw_orders_df.filter(
    col("_corrupt_record").isNotNull()
)
display(quarantine_orders_df)

valid_orders_df = raw_orders_df.filter(
    col("_corrupt_record").isNull()
)
display(valid_orders_df)

In [0]:
quarantine_customers_df = raw_customers_df.filter(
    col("_corrupt_record").isNotNull()
)
display(quarantine_customers_df)

valid_customers_df = raw_customers_df.filter(
    col("_corrupt_record").isNull()
)
display(valid_customers_df)

In [0]:
quarantine_products_df = raw_products_df.filter(
    col("_corrupt_record").isNotNull()
)
display(quarantine_products_df)

valid_products_df = raw_products_df.filter(
    col("_corrupt_record").isNull()
)
display(valid_products_df)

## Phase 2 — Cleaning & Standardisation

- Rename columns across DataFrames to follow a consistent snake_case naming convention
- Cast columns to their appropriate data types (prices are strings, dates are strings, etc.)
- Drop irrelevant columns that the analytics team has confirmed they don't need
- Handle nulls strategically — fill some with defaults, drop rows where critical fields are missing
- Remove duplicate customer records, keeping the most recently registered entry

- ### Rename columns across DataFrames to follow a consistent snake_case naming convention
- ### Cast columns to their appropriate data types
- ### Standardise string fields — trim whitespace, convert emails to lowercase, clean up the currency symbols from price fields

In [0]:
standardized_orders_df = valid_orders_df\
                            .withColumn("product_ids", split("product_ids", ","))\
                            .withColumn("order_date", to_date("order_date", "yyyy-MM-dd"))\
                            .withColumn("shipping_date", to_date("shipping_date", "yyyy-MM-dd"))\
                            .withColumn("amount", col("amount").cast(DoubleType()))\
                            .withColumn("discount_tags", split("discount_tags", ","))

display(standardized_orders_df)

In [0]:
standardized_customers_df = valid_customers_df\
                            .withColumnRenamed("FullName", "full_name")\
                            .withColumnRenamed("EMAIL", "email")\
                            .withColumnRenamed("City", "city")\
                            .withColumnRenamed("Country", "country")\
                            .withColumnRenamed("Phone", "phone")\
                            .withColumn("registration_date", to_date("registration_date", "yyyy-MM-dd"))\
                            .withColumn("email", trim(lower("email")))\
                            .withColumn("full_name", trim("full_name"))
                            
display(standardized_customers_df)

In [0]:
standardized_products_df = valid_products_df\
                            .withColumnRenamed("ProductName", "product_name")\
                            .withColumnRenamed("Category", "category")\
                            .withColumnRenamed("Price", "price")\
                            .withColumn("price", regexp_replace("price", "₹", "").cast(DoubleType()))\
                            .withColumn("tags", split("tags", ","))

display(standardized_products_df)

- ### Drop irrelevant columns that the analytics team has confirmed they don't need
- ### Handle nulls strategically — fill some with defaults, drop rows where critical fields are missing
- ### Remove duplicate customer records, keeping the most recently registered entry

In [0]:
cleaned_orders_df = standardized_orders_df\
                    .drop("_corrupt_record")\
                    .dropDuplicates(["order_id"])\
                    .dropna(subset=["order_id", "customer_id", "product_ids", "order_date", "amount", "status"], how="any")\
                    .withColumn(
                        "discount_tags",
                        when(col("discount_tags").isNull(), array(lit("NONE")))
                        .otherwise(col("discount_tags"))
                    )

display(cleaned_orders_df)

In [0]:
cleaned_customers_df = standardized_customers_df\
                    .drop("phone", "country", "_corrupt_record")\
                    .sort(col("customer_id").asc(), col("registration_date").desc())\
                    .dropDuplicates(["customer_id"])\
                    .dropna(subset=["customer_id", "email", "registration_date"], how="any")\
                    .fillna("Unknown", subset=["full_name", "city"])

display(cleaned_customers_df)                    

In [0]:
cleaned_products_df = standardized_products_df\
                    .drop("_corrupt_record")\
                    .dropDuplicates(["product_id"])\
                    .dropna(subset=["product_id", "product_name", "price"], how="any")\
                    .fillna("Uncategorised", subset="category")\
                    .withColumn(
                        "tags",
                        when(col("tags").isNull(), array(lit("NONE")))
                        .otherwise(col("tags"))
                    )

display(cleaned_products_df)

## Phase 3 — Transformations & Enrichment

- Create new derived columns such as order year, order month, the number of days taken to ship, and a customer name split into first and last name
- Flag orders as "High Value" or "Standard" based on the order amount using a new column
- Sort the orders DataFrame in a meaningful way for downstream reporting
- Apply a limit where appropriate for creating sample/preview datasets

In [0]:
enriched_orders_df = cleaned_orders_df\
                    .withColumn("order_year", year("order_date"))\
                    .withColumn("order_month", month("order_date"))\
                    .withColumn("days_to_ship", datediff("shipping_date", "order_date"))\
                    .withColumn(
                        "order_category",
                        when(col("amount") >= HIGH_VALUE_THRESHOLD, "High Value")
                        .otherwise("Standard")
                    )\
                    .sort(col("order_date").asc(), col("customer_id").asc())

display(enriched_orders_df)

In [0]:
enriched_customers_df = cleaned_customers_df\
                        .withColumn("first_name", element_at(split("full_name", " "), 1))\
                        .withColumn(
                            "last_name",
                            when(size(split("full_name", " ")) == 1, lit("Unknown"))
                            .otherwise(element_at(split("full_name", " "), 2))
                        )\
                        .sort(col("customer_id").asc())

display(enriched_customers_df)

In [0]:
enriched_products_df = cleaned_products_df\
                        .sort(col("category").asc(), col("price").asc())

display(enriched_products_df)      

In [0]:
orders_preview_df = enriched_orders_df.limit(5)
display(orders_preview_df)

customers_preview_df = enriched_customers_df.limit(5)
display(customers_preview_df)

products_preview_df = enriched_products_df.limit(5)
display(products_preview_df)

## Phase 4 — Advanced Transformations

- The product IDs purchased in each order are stored as a comma-separated string — split and explode this so that each row represents one order-product combination
- Similarly, explode the product tags in the products file so each tag gets its own row
- After exploding, check whether a specific tag or product category is present in an order using array functions
- Combine customer data from two regional source files (which may have different column ordering) into a single unified DataFrame

In [0]:
adv_transformations_orders_df = enriched_orders_df\
                                .withColumn("product_id", explode("product_ids"))\
                                .drop("product_ids")\
                                .withColumn(
                                    "is_sale_order",
                                    array_contains(col("discount_tags"), "SALE")
                                )

display(adv_transformations_orders_df)

In [0]:
adv_transformations_products_df = enriched_products_df\
                                .withColumn(
                                    "is_bluetooth_product",
                                    array_contains(col("tags"), "bluetooth")
                                )\
                                .withColumn("tag", explode("tags"))\
                                .drop("tags")
    
display(adv_transformations_products_df)

### Combine customer data from two regional source files (which may have different column ordering) into a single unified DataFrame

In [0]:
north_customers_df = cleaned_customers_df.filter(col("city").isin(NORTH_INDIAN_CITIES))

south_customers_df = cleaned_customers_df.filter(col("city").isin(SOUTH_INDIAN_CITIES))

In [0]:
# North has standard order
# customer_id, full_name, email, city, registration_date

# South has different column order (as if from a different source system)
south_customers_df = south_customers_df\
                    .select(
                        "email", "city", "customer_id", "full_name", "registration_date"
                    )

In [0]:
combined_customers_df = north_customers_df.unionByName(south_customers_df)

display(combined_customers_df)

## Phase 5 — Final Output

- Join the cleaned and exploded DataFrames together to produce a master orders report that the analytics team can query
- Select only the final required columns and present a clean, analytics-ready DataFrame

In [0]:
master_df = adv_transformations_orders_df\
            .join(adv_transformations_products_df, on="product_id", how="left")\
            .join(enriched_customers_df, on="customer_id", how="left")\
            .select(
                "order_id", "order_date", "order_year", "order_month",
                "status", "amount", "days_to_ship", "order_category", 
                "is_sale_order", "customer_id", "full_name", "email", 
                "city", "product_id", "product_name", "category", "price", 
                "tag", "is_bluetooth_product"
            )

display(master_df)